In [29]:
import numpy as np
import pandas as pd
import pyarrow as pa

# **Avoid `dtype=object`**

In [4]:
ser_obj = pd.Series(["foo", "bar", "baz"] * 10_000, dtype=object)
ser_str = pd.Series(["foo", "bar", "baz"] * 10_000, dtype=pd.StringDtype())

In [6]:
# ser_str.iloc[0] = False => TypeError: Invalid value 'False' for dtype 'string'. Value should be a string or missing value, 
# got 'bool' instead.

In [7]:
ser_obj.iloc[0] = False

In [9]:
ser_obj.str.capitalize().head() # in object data type the boolean value was convert to missing value

0    NaN
1    Bar
2    Baz
3    Foo
4    Bar
dtype: object

In [10]:
ser_obj = pd.Series(["foo", "bar", "baz"] * 10_000, dtype=object)
ser_str = pd.Series(["foo", "bar", "baz"] * 10_000, dtype=pd.StringDtype())

In [11]:
import timeit

In [17]:
timeit.timeit(ser_obj.str.upper, number=1_000)

4.019790697999952

In [18]:
timeit.timeit(ser_str.str.upper, number=1_000)

4.048010822999913

In [20]:
import io

In [21]:
data = io.StringIO("int_col,string_col\n0,foo\n1,bar\n2,baz")
data.seek(0)
pd.read_csv(data, dtype_backend="numpy_nullable").dtypes

int_col                Int64
string_col    string[python]
dtype: object

In [23]:
df = pd.DataFrame([
    [0, "foo"],
    [1, "bar"],
    [2, "baz"],
], columns=["int_col", "string_col"]).convert_dtypes(dtype_backend="numpy_nullable")

df.dtypes

int_col                Int64
string_col    string[python]
dtype: object

In [24]:
import datetime

In [25]:
ser = pd.Series([
    datetime.date(2024, 1, 1),
    datetime.date(2024, 1, 2),
    datetime.date(2024, 1, 3),
])

ser

0    2024-01-01
1    2024-01-02
2    2024-01-03
dtype: object

In [27]:
# ser.dt.year => AttributeError: Can only use .dt accessor with datetimelike values

In [30]:
ser = pd.Series(
    [datetime.date(2024, 1, 1),
    datetime.date(2024, 1, 2),
    datetime.date(2024, 1, 3),
], dtype=pd.ArrowDtype(pa.date32()))

ser

0    2024-01-01
1    2024-01-02
2    2024-01-03
dtype: date32[day][pyarrow]

In [31]:
ser.dt.year

0    2024
1    2024
2    2024
dtype: int64[pyarrow]

# **Be cognizant of data sizes**

In [32]:
df = pd.DataFrame({
    "a": [0] * 100_000,
    "b": [2 ** 8] * 100_000,
    "c": [2 ** 16] * 100_000,
    "d": [2 ** 32] * 100_000,
}).convert_dtypes(dtype_backend="numpy_nullable")

df.head()

,a,b,c,d
0,0,256,65536,4294967296
1,0,256,65536,4294967296
2,0,256,65536,4294967296
3,0,256,65536,4294967296
4,0,256,65536,4294967296


In [34]:
df.memory_usage() # total Bytes reserved in each column

Index       132
a        900000
b        900000
c        900000
d        900000
dtype: int64

In [35]:
df.assign(
    a=lambda x: x["a"].astype(pd.Int8Dtype()),
    b=lambda x: x["b"].astype(pd.Int16Dtype()),
    c=lambda x: x["c"].astype(pd.Int32Dtype()),
).memory_usage()

Index       132
a        200000
b        300000
c        500000
d        900000
dtype: int64

In [39]:
df.select_dtypes("number").assign(
    **{x: pd.to_numeric(
        y, 
        downcast="signed", 
        dtype_backend="numpy_nullable"
    ) for x, y in df.items()}
).memory_usage()

Index       132
a        200000
b        300000
c        500000
d        900000
dtype: int64

# **Use vectorized functions instead of loops**

In [41]:
ser = pd.Series(range(100_000), dtype=pd.Int64Dtype())

In [43]:
ser.sum()

np.int64(4999950000)

In [44]:
result = 0

for n in ser:
    result += n

result

np.int64(4999950000)

In [45]:
timeit.timeit(ser.sum, number=1000)

0.12649774300007266

In [46]:
def loop_sum():
    result = 0
    for x in ser: result += x

In [47]:
timeit.timeit(loop_sum, number=1000)

9.419441664999795

In [49]:
df = pd.DataFrame({
    "column": ["a", "a", "b", "a", "b"],
    "value": [0, 1, 2, 4, 8],
}).convert_dtypes(dtype_backend="numpy_nullable")

for label, group in df.groupby("column"): # using the loop only when deal in whith The GroupBy
    print(f"The group for label {label} is:\n{group}\n")

The group for label a is:
  column  value
0      a      0
1      a      1
3      a      4

The group for label b is:
  column  value
2      b      2
4      b      8



# **Avoid mutating data**

In [50]:
def mutate_after():
    data = ["foo", "bar", "baz"]
    ser = pd.Series(data, dtype=pd.StringDtype())
    ser.iloc[1] = "BAR"

In [51]:
timeit.timeit(mutate_after, number=1000)

0.11751774300046236

In [52]:
def mutate_before():
    data = ["foo", "bar", "baz"]
    data[1] = "BAR"
    ser = pd.Series(data, dtype=pd.StringDtype())

In [54]:
timeit.timeit(mutate_before, number=1_000) # we should mutate data before we load it in a pandas structure 

0.06653174599978229

# **Dictionary-encode low cardinality data**

In [55]:
values = ["foo", "bar", "baz"]
ser = pd.Series(values * 100_000, dtype=pd.StringDtype())

ser.memory_usage()

2400132

In [56]:
cat = pd.CategoricalDtype(values)
ser = pd.Series(values * 100_000, dtype=cat)

ser.memory_usage()

300264

# **Test-driven development features**

In [57]:
import unittest

In [58]:
class MyTests(unittest.TestCase):
    def test_42(self):
        self.assertEqual(21 * 2, 42)

In [59]:
def suite():
    suite = unittest.TestSuite()
    suite.addTest(MyTests("test_42"))
    return suite

In [60]:
runner = unittest.TextTestRunner()
runner.run(suite())

.
----------------------------------------------------------------------
Ran 1 test in 0.004s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

In [61]:
def some_cool_numbers():
    return pd.Series([42, 555, pd.NA], dtype=pd.Int64Dtype())
class MyTests(unittest.TestCase):
    def test_cool_numbers(self):
        result = some_cool_numbers()
        expected = pd.Series([42, 555, pd.NA], dtype=pd.Int64Dtype())
        self.assertEqual(result, expected)

In [62]:
def suite():
    suite = unittest.TestSuite()
    suite.addTest(MyTests("test_cool_numbers"))
    return suite

In [63]:
runner = unittest.TextTestRunner()
runner.run(suite())

E
ERROR: test_cool_numbers (__main__.MyTests.test_cool_numbers)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/tmp/ipykernel_4963/1227123780.py", line 7, in test_cool_numbers
    self.assertEqual(result, expected)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^
  File "/home/el7m7/anaconda3/lib/python3.13/unittest/case.py", line 907, in assertEqual
    assertion_func(first, second, msg=msg)
    ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/el7m7/anaconda3/lib/python3.13/unittest/case.py", line 897, in _baseAssertEqual
    if not first == second:
           ^^^^^^^^^^^^^^^
  File "/home/el7m7/anaconda3/lib/python3.13/site-packages/pandas/core/generic.py", line 1580, in __nonzero__
    raise ValueError(
    ...<2 lines>...
    )
ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

----------------------------------------------------------------------
Ran 1 test in 0.01

<unittest.runner.TextTestResult run=1 errors=1 failures=0>

In [64]:
result = some_cool_numbers()
expected = pd.Series([42, 555, pd.NA], dtype=pd.Int64Dtype())
result == expected

0    True
1    True
2    <NA>
dtype: boolean

In [65]:
import pandas.testing as tm

In [66]:
def some_cool_numbers():
    return pd.Series([42, 555, pd.NA], dtype=pd.Int64Dtype())

class MyTests(unittest.TestCase):
    def test_cool_numbers(self):
        result = some_cool_numbers()
        expected = pd.Series([42, 555, pd.NA], dtype=pd.Int64Dtype())
        tm.assert_series_equal(result, expected)

In [67]:
def suite():
    suite = unittest.TestSuite()
    suite.addTest(MyTests("test_cool_numbers"))
    return suite

In [68]:
runner = unittest.TextTestRunner()
runner.run(suite())

.
----------------------------------------------------------------------
Ran 1 test in 0.006s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

In [69]:
def some_cool_numbers():
    return pd.Series([42, 555, pd.NA], dtype=pd.Int64Dtype())

class MyTests(unittest.TestCase):
    def test_cool_numbers(self):
        result = some_cool_numbers()
        expected = pd.Series([42, 555, pd.NA], dtype=pd.Int32Dtype())
        tm.assert_series_equal(result, expected)

In [70]:
def suite():
    suite = unittest.TestSuite()
    suite.addTest(MyTests("test_cool_numbers"))
    return suite

In [71]:
runner = unittest.TextTestRunner()
runner.run(suite())

F
FAIL: test_cool_numbers (__main__.MyTests.test_cool_numbers)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/tmp/ipykernel_4963/3014089158.py", line 8, in test_cool_numbers
    tm.assert_series_equal(result, expected)
    ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^
  File "/home/el7m7/anaconda3/lib/python3.13/site-packages/pandas/_testing/asserters.py", line 999, in assert_series_equal
    assert_attr_equal("dtype", left, right, obj=f"Attributes of {obj}")
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/el7m7/anaconda3/lib/python3.13/site-packages/pandas/_testing/asserters.py", line 421, in assert_attr_equal
    raise_assert_detail(obj, msg, left_attr, right_attr)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/el7m7/anaconda3/lib/python3.13/site-packages/pandas/_testing/asserters.py", line 620, in raise_assert_detail
    raise AssertionError(msg)
AssertionErro

<unittest.runner.TextTestResult run=1 errors=0 failures=1>